In [1]:
import torch

if torch.cuda.is_available():
    device = 0
    prop = torch.cuda.get_device_properties(device)
    print(f"Allocated GPU Name: {prop.name}")
    print(f"Total Available VRAM: {prop.total_memory / (1024**2):.0f} MiB")
    print(f"Compute Capability Major.Minor: {prop.major}.{prop.minor}")

    # Query max threads per block via PyTorch device attributes
    max_threads = torch.cuda.get_device_properties(device).max_threads_per_block
    print(f"Max Threads Per Block: {max_threads}")
else:
    print("CUDA is not available.")

Fri Oct  9 05:34:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [11]:
for N in [100, 1_000, 10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]:
    cpu, total, kernel = benchmark_run(N)
    print(f"{N:>10,} | CPU {cpu:10.3f} | GPU total {total:10.3f} | GPU kernel {kernel:10.3f}")

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


       100 | CPU      0.074 | GPU total      1.417 | GPU kernel      0.082
     1,000 | CPU      0.248 | GPU total      0.702 | GPU kernel      0.068
    10,000 | CPU      2.459 | GPU total      0.559 | GPU kernel      0.058
   100,000 | CPU     25.493 | GPU total      0.659 | GPU kernel      0.072
 1,000,000 | CPU    271.441 | GPU total      4.617 | GPU kernel      0.134
 5,000,000 | CPU        nan | GPU total     14.539 | GPU kernel      0.276
10,000,000 | CPU        nan | GPU total     25.410 | GPU kernel      0.532


In [13]:
import numpy as np
import time
from numba import cuda

# 1. CPU Reference Implementation (Pure Python loop)
def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2.0
    return out

# 2. GPU Kernel Implementation
@cuda.jit
def double_gpu_kernel(d_in, d_out):
    # Calculate unique 1D thread ID
    idx = cuda.grid(1)
    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0

In [14]:
def benchmark_run(N, student_id_seed=1234):
    np.random.seed(student_id_seed)
    h_in = np.random.rand(N).astype(np.float32)

    # Measure CPU Pure Loop (skip for N > 1M, too slow)
    if N <= 1_000_000:
        t0 = time.perf_counter()
        _ = double_cpu_pure(h_in)
        cpu_time_ms = (time.perf_counter() - t0) * 1000.0
    else:
        cpu_time_ms = float('nan')  # Extrapolate or skip

    # GPU Setup
    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

    # Warmup compilation run
    d_warmup = cuda.to_device(np.ones(10, dtype=np.float32))
    double_gpu_kernel[1, 32](d_warmup, d_warmup)
    cuda.synchronize()

    # GPU Roundtrip Measurement (Includes PCIe transfers)
    t0 = time.perf_counter()
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    h_out = d_out.copy_to_host()
    cuda.synchronize()
    gpu_total_ms = (time.perf_counter() - t0) * 1000.0

    # GPU Compute-Only Measurement (Pre-allocated in VRAM)
    t0 = time.perf_counter()
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    cuda.synchronize()
    gpu_compute_ms = (time.perf_counter() - t0) * 1000.0

    return cpu_time_ms, gpu_total_ms, gpu_compute_ms

In [15]:
import numpy as np
from numba import cuda

# DEFECTIVE KERNEL: note the missing 'if idx < n' check!
@cuda.jit
def defective_kernel(d_arr):
    idx = cuda.grid(1)
    # FORBIDDEN: writing without a bounds check!
    d_arr[idx] = 999.0

# TEST HARNESS
N = 1000
h_arr = np.zeros(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)

# Launch 4 blocks of 256 = 1024 threads
threads_per_block = 256
blocks_per_grid = 4

print("Launching defective kernel with 1024 threads on 1000-element array...")
defective_kernel[blocks_per_grid, threads_per_block](d_arr)
cuda.synchronize()
print("Execution finished (or did it crash?). Checking host copy...")

h_result = d_arr.copy_to_host()
print(f"Elements processed: {len(h_result)}. Last element: {h_result[-1]}")

Launching defective kernel with 1024 threads on 1000-element array...
Execution finished (or did it crash?). Checking host copy...
Elements processed: 1000. Last element: 999.0


In [16]:
import numpy as np
from numba import cuda

@cuda.jit
def inspect_threads_kernel(log_block, log_thread, log_global, log_active, N):
    t_idx = cuda.threadIdx.x
    b_idx = cuda.blockIdx.x
    g_idx = cuda.grid(1)
    if g_idx < log_block.shape[0]:
        log_block[g_idx] = b_idx
        log_thread[g_idx] = t_idx
        log_global[g_idx] = g_idx
        log_active[g_idx] = 1 if g_idx < N else 0

# Configuration
N = 37
threads_per_block = 8
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block  # 5 blocks = 40 threads

total_threads = blocks_per_grid * threads_per_block
log_b = cuda.device_array(total_threads, dtype=np.int32)
log_t = cuda.device_array(total_threads, dtype=np.int32)
log_g = cuda.device_array(total_threads, dtype=np.int32)
log_a = cuda.device_array(total_threads, dtype=np.int32)

inspect_threads_kernel[blocks_per_grid, threads_per_block](log_b, log_t, log_g, log_a, N)
cuda.synchronize()

# Copy to host once, so the loop doesn't do a PCIe read per element
hb, ht, hg, ha = (a.copy_to_host() for a in (log_b, log_t, log_g, log_a))

print(f"{'Global ID':<10} | {'Block ID':<10} | {'Thread ID':<10} | {'Status':<10}")
print("-" * 50)
for i in range(32, 40):  # Block 4, the boundary transition
    status = "ACTIVE" if ha[i] == 1 else "IDLE (GUARDED)"
    print(f"{hg[i]:<10} | {hb[i]:<10} | {ht[i]:<10} | {status:<10}")

Global ID  | Block ID   | Thread ID  | Status    
--------------------------------------------------
32         | 4          | 0          | ACTIVE    
33         | 4          | 1          | ACTIVE    
34         | 4          | 2          | ACTIVE    
35         | 4          | 3          | ACTIVE    
36         | 4          | 4          | ACTIVE    
37         | 4          | 5          | IDLE (GUARDED)
38         | 4          | 6          | IDLE (GUARDED)
39         | 4          | 7          | IDLE (GUARDED)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [17]:
import numpy as np
from numba import cuda
import time

@cuda.jit
def naive_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        d_total[0] += d_arr[idx]

@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        cuda.atomic.add(d_total, 0, d_arr[idx])

N = 50_000
tpb = 256
bpg = (N + tpb - 1) // tpb
d_arr = cuda.to_device(np.ones(N, dtype=np.float32))

for trial in range(1, 6):
    d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
    naive_sum_kernel[bpg, tpb](d_arr, d_total)
    cuda.synchronize()
    out = d_total.copy_to_host()[0]
    print(f"Trial {trial}: {out}  (lost updates: {N - out})")

d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
atomic_sum_kernel[bpg, tpb](d_arr, d_total)  # warmup/compile
cuda.synchronize()

d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
t0 = time.perf_counter()
atomic_sum_kernel[bpg, tpb](d_arr, d_total)
cuda.synchronize()
print("Atomic:", d_total.copy_to_host()[0], f"time {(time.perf_counter()-t0)*1000:.3f} ms")

Trial 1: 2.0  (lost updates: 49998.0)
Trial 2: 2.0  (lost updates: 49998.0)
Trial 3: 2.0  (lost updates: 49998.0)
Trial 4: 2.0  (lost updates: 49998.0)
Trial 5: 2.0  (lost updates: 49998.0)
Atomic: 50000.0 time 0.585 ms


In [19]:
import numpy as np
from numba import cuda

@cuda.jit
def inspect_threads_kernel(log_block, log_thread, log_global, log_active, N):
    t_idx = cuda.threadIdx.x
    b_idx = cuda.blockIdx.x
    g_idx = cuda.grid(1)
    if g_idx < log_block.shape[0]:
        log_block[g_idx] = b_idx
        log_thread[g_idx] = t_idx
        log_global[g_idx] = g_idx
        log_active[g_idx] = 1 if g_idx < N else 0

# Configuration
N = 37
threads_per_block = 8
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block  # 5 blocks = 40 threads

total_threads = blocks_per_grid * threads_per_block
log_b = cuda.device_array(total_threads, dtype=np.int32)
log_t = cuda.device_array(total_threads, dtype=np.int32)
log_g = cuda.device_array(total_threads, dtype=np.int32)
log_a = cuda.device_array(total_threads, dtype=np.int32)

inspect_threads_kernel[blocks_per_grid, threads_per_block](log_b, log_t, log_g, log_a, N)
cuda.synchronize()

hb, ht, hg, ha = (a.copy_to_host() for a in (log_b, log_t, log_g, log_a))

print(f"{'Global ID':<10} | {'Block ID':<10} | {'Thread ID':<10} | {'Status':<10}")
print("-" * 50)
for i in range(32, 40):
    status = "ACTIVE" if ha[i] == 1 else "IDLE (GUARDED)"
    print(f"{hg[i]:<10} | {hb[i]:<10} | {ht[i]:<10} | {status:<10}")


Global ID  | Block ID   | Thread ID  | Status    
--------------------------------------------------
32         | 4          | 0          | ACTIVE    
33         | 4          | 1          | ACTIVE    
34         | 4          | 2          | ACTIVE    
35         | 4          | 3          | ACTIVE    
36         | 4          | 4          | ACTIVE    
37         | 4          | 5          | IDLE (GUARDED)
38         | 4          | 6          | IDLE (GUARDED)
39         | 4          | 7          | IDLE (GUARDED)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
